# 02 - Paired 44-fold cross-validation

Baseline against GA-optimised, 40 epochs per run, 88 runs in total.
Produces `cv_results.csv`, the input of `analysis_paired_tests.py`.

The search ran across several Colab sessions. Because every split is seeded at 42 and drawn
over group names, fold *i* is the same fold in every session, so results from separate
sessions join legitimately. The restore cell below carries the folds completed earlier.

## Session setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

if not os.path.exists('/content/santos_sss'):
    os.system('unzip -q /content/drive/MyDrive/santos_sss.zip -d /content/santos_sss')

os.makedirs('/content/working', exist_ok=True)
os.makedirs('/kaggle', exist_ok=True)
if not os.path.exists('/kaggle/working'):
    os.symlink('/content/working', '/kaggle/working')

DRIVE_BAK = '/content/drive/MyDrive/santos_backup'
os.makedirs(DRIVE_BAK, exist_ok=True)

import glob
n = len(glob.glob('/content/santos_sss/**/*.jpg', recursive=True))
print(f"Dataset images: {n} (πρέπει: 1170)")
print("Symlink OK:", os.path.realpath('/kaggle/working'))

## Environment

In [ ]:
import os, glob, shutil, random, json
import numpy as np, pandas as pd

if not os.path.exists('/kaggle/working/yolov5'):
    os.system('git clone https://github.com/ultralytics/yolov5.git /kaggle/working/yolov5')
    os.system('pip install -r /kaggle/working/yolov5/requirements.txt')
    os.system('pip install albumentations')

os.chdir('/kaggle/working/yolov5')

BASE = '/content/santos_sss'
if not os.path.exists(BASE):
    raise FileNotFoundError("Τρέξε πρώτα το Cell 0 (Colab setup)")
print("Setup OK. Dataset base:", BASE)

## Offline geometric augmentation

In [ ]:
import cv2
import albumentations as A
random.seed(42); np.random.seed(42)

SRC   = BASE
AUG   = '/kaggle/working/santos_augmented'
N_AUG = 3

if os.path.exists(AUG): shutil.rmtree(AUG)
os.makedirs(f'{AUG}/images', exist_ok=True)
os.makedirs(f'{AUG}/labels', exist_ok=True)

def label_for(img_path):
    lbl = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    if not os.path.exists(lbl):
        lbl = img_path.rsplit('.', 1)[0] + '.txt'
    return lbl if os.path.exists(lbl) else None

def read_yolo(lbl_path):
    boxes, classes = [], []
    if lbl_path:
        for line in open(lbl_path):
            p = line.split()
            if len(p) == 5:
                x, y, w, h = (float(v) for v in p[1:])
                x = min(max(x, 0.0), 1.0)
                y = min(max(y, 0.0), 1.0)
                w = min(w, 2*x, 2*(1-x))
                h = min(h, 2*y, 2*(1-y))
                if w <= 0 or h <= 0:
                    continue
                classes.append(int(p[0]))
                boxes.append([x, y, w, h])
    return boxes, classes

def write_yolo(lbl_path, boxes, classes):
    with open(lbl_path, 'w') as f:
        for c, b in zip(classes, boxes):
            f.write(f"{c} {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}\n")

transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Affine(rotate=(-10, 10), translate_percent=(-0.10, 0.10),
             scale=(0.8, 1.2), border_mode=cv2.BORDER_CONSTANT, p=0.9),
], bbox_params=A.BboxParams(format='yolo', label_fields=['classes'],
                            min_visibility=0.3))

imgs = sorted(glob.glob(os.path.join(SRC, '**', '*.jpg'), recursive=True))
print(f"Found {len(imgs)} originals. Target: {len(imgs)*(1+N_AUG)} total.")

groups = {}
def save_pair(img, boxes, classes, stem, group_stem):
    cv2.imwrite(f'{AUG}/images/{stem}.jpg', img)
    write_yolo(f'{AUG}/labels/{stem}.txt', boxes, classes)
    groups[stem] = group_stem

for img_path in imgs:
    stem = os.path.basename(img_path).rsplit('.', 1)[0]
    img  = cv2.imread(img_path)
    boxes, classes = read_yolo(label_for(img_path))
    save_pair(img, boxes, classes, stem, stem)
    made = tries = 0
    while made < N_AUG and tries < N_AUG * 5:
        tries += 1
        t = transform(image=img, bboxes=boxes, classes=classes)
        if boxes and len(t['bboxes']) == 0:
            continue
        save_pair(t['image'], [list(b) for b in t['bboxes']],
                  list(t['classes']), f'{stem}_aug{made}', stem)
        made += 1

json.dump(groups, open(f'{AUG}/groups.json', 'w'))
print(f"Done. Total files: {len(glob.glob(f'{AUG}/images/*.jpg'))}")
print(f"Group map saved -> {AUG}/groups.json")

## Hold-out and 44 stratified group-aware folds

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split
random.seed(42); np.random.seed(42)

AUG = '/kaggle/working/santos_augmented'
groups = json.load(open(f'{AUG}/groups.json'))
all_imgs = sorted(glob.glob(f'{AUG}/images/*.jpg'))

def content_label(img_path):
    lbl = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    hm = hn = False
    if os.path.exists(lbl):
        for line in open(lbl):
            line = line.strip()
            if not line: continue
            c = int(line.split()[0])
            if c == 0: hm = True
            elif c == 1: hn = True
    if hm and hn: return 3
    if hm:        return 1
    if hn:        return 2
    return 0

group_ids   = sorted(set(groups.values()))
group_label = {g: content_label(f'{AUG}/images/{g}.jpg') for g in group_ids}

g_arr = np.array(group_ids)
y_arr = np.array([group_label[g] for g in group_ids])
print("Groups:", len(g_arr), "| label counts:",
      {int(k): int((y_arr==k).sum()) for k in np.unique(y_arr)})

tv_groups, test_groups = train_test_split(
    g_arr, test_size=0.10, stratify=y_arr, random_state=42)
test_groups = set(test_groups); tv_groups = set(tv_groups)

def files_of(group_set):
    return np.array([p for p in all_imgs
                     if groups[os.path.basename(p).rsplit('.',1)[0]] in group_set])

test_imgs = files_of(test_groups)
tv_imgs   = files_of(tv_groups)
print(f"Hold-out test files: {len(test_imgs)} | train+val files: {len(tv_imgs)}")

tv_group_list = sorted(tv_groups)
tv_y = np.array([group_label[g] for g in tv_group_list])
skf = StratifiedKFold(n_splits=44, shuffle=True, random_state=42)

tv_group_of = np.array([groups[os.path.basename(p).rsplit('.',1)[0]] for p in tv_imgs])
folds = []
for tr_g_idx, va_g_idx in skf.split(tv_group_list, tv_y):
    tr_groups = set(np.array(tv_group_list)[tr_g_idx])
    va_groups = set(np.array(tv_group_list)[va_g_idx])
    tr_mask = np.array([g in tr_groups for g in tv_group_of])
    va_mask = np.array([g in va_groups for g in tv_group_of])
    folds.append((np.where(tr_mask)[0], np.where(va_mask)[0]))

print(f"Built {len(folds)} folds. Example fold sizes: "
      f"train={len(folds[0][0])}, val={len(folds[0][1])}")

## Hyperparameters, helpers and fold runner

In [ ]:
EPOCHS_CV = 40
IMG       = 512
results_path = '/kaggle/working/cv_results.csv'

ga_hyp = 'lr0: 0.004371\nlrf: 0.083805\nmomentum: 0.937932\nweight_decay: 0.000639\nwarmup_epochs: 3.0\nwarmup_momentum: 0.8\nwarmup_bias_lr: 0.1\nbox: 0.045784\ncls: 0.240395\ncls_pw: 1.0\nobj: 1.948448\nobj_pw: 1.0\niou_t: 0.20\nanchor_t: 4.0\nfl_gamma: 0.0\nhsv_h: 0.0\nhsv_s: 0.0\nhsv_v: 0.0\ndegrees: 0.0\ntranslate: 0.0\nscale: 0.0\nshear: 0.0\nperspective: 0.0\nflipud: 0.0\nfliplr: 0.0\nmosaic: 0.0\nmixup: 0.0\ncopy_paste: 0.0\n'
with open('/kaggle/working/hyp_ga_best.yaml','w') as f: f.write(ga_hyp)

baseline_hyp = 'lr0: 0.01\nlrf: 0.01\nmomentum: 0.937\nweight_decay: 0.0005\nwarmup_epochs: 3.0\nwarmup_momentum: 0.8\nwarmup_bias_lr: 0.1\nbox: 0.05\ncls: 0.5\ncls_pw: 1.0\nobj: 1.0\nobj_pw: 1.0\niou_t: 0.20\nanchor_t: 4.0\nfl_gamma: 0.0\nhsv_h: 0.0\nhsv_s: 0.0\nhsv_v: 0.0\ndegrees: 0.0\ntranslate: 0.0\nscale: 0.0\nshear: 0.0\nperspective: 0.0\nflipud: 0.0\nfliplr: 0.0\nmosaic: 0.0\nmixup: 0.0\ncopy_paste: 0.0\n'
with open('/kaggle/working/hyp_baseline.yaml','w') as f: f.write(baseline_hyp)

GA_HYP   = '/kaggle/working/hyp_ga_best.yaml'
BASE_HYP = '/kaggle/working/hyp_baseline.yaml'

def empty_txt_for_backgrounds(img_list, labels_dir):
    for p in img_list:
        stem = os.path.basename(p).rsplit('.', 1)[0]
        src  = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        dst  = os.path.join(labels_dir, stem + '.txt')
        if os.path.exists(src): shutil.copy(src, dst)
        else: open(dst, 'w').close()

def make_split_dir(name, train_imgs, val_imgs):
    root = f'/kaggle/working/data_{name}'
    if os.path.exists(root): shutil.rmtree(root)
    for sub in ['images/train','images/val','labels/train','labels/val']:
        os.makedirs(os.path.join(root, sub), exist_ok=True)
    for p in train_imgs: shutil.copy(p, os.path.join(root,'images/train',os.path.basename(p)))
    for p in val_imgs:   shutil.copy(p, os.path.join(root,'images/val',os.path.basename(p)))
    empty_txt_for_backgrounds(train_imgs, os.path.join(root,'labels/train'))
    empty_txt_for_backgrounds(val_imgs,   os.path.join(root,'labels/val'))
    yp = os.path.join(root,'data.yaml')
    with open(yp,'w') as f:
        f.write(f"train: {root}/images/train\nval: {root}/images/val\nnc: 2\nnames: ['MILCO','NOMBO']\n")
    return yp

def parse_results(run_dir):
    df = pd.read_csv(os.path.join(run_dir,'results.csv'))
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1]
    P = float(last['metrics/precision']); R = float(last['metrics/recall'])
    f1 = 2*P*R/(P+R) if (P+R) > 0 else 0.0
    return {'precision': P, 'recall': R, 'f1': f1,
            'mAP50': float(last['metrics/mAP_0.5']),
            'mAP50_95': float(last['metrics/mAP_0.5:0.95'])}

def already_done(config, fold):
    if not os.path.exists(results_path): return False
    d = pd.read_csv(results_path)
    return ((d['config']==config) & (d['fold']==fold)).any()

def save_row(m):
    rows = pd.read_csv(results_path).to_dict('records') if os.path.exists(results_path) else []
    rows.append(m); pd.DataFrame(rows).to_csv(results_path, index=False)

def run_one(config, fold, epochs=EPOCHS_CV):
    if already_done(config, fold):
        print(f"  {config} fold{fold}: already done - skip"); return
    tr, va = folds[fold]
    yp = make_split_dir(f'fold{fold}', tv_imgs[tr], tv_imgs[va])
    hyp = GA_HYP if config == 'ga' else BASE_HYP
    name = f'{config}_fold{fold}'
    log  = f'/kaggle/working/log_{name}.txt'
    cmd = (f'python train.py --img {IMG} --batch 16 --epochs {epochs} --data {yp} '
           f'--weights yolov5s.pt --hyp {hyp} --name {name} --project /kaggle/working/runs '
           f'--exist-ok --cache > {log} 2>&1')
    print(f"  running {name} ... -> {log}  (~30 min)")
    os.system(cmd)
    rc = f'/kaggle/working/runs/{name}/results.csv'
    if not os.path.exists(rc):
        print(f"  !! {name} FAILED. Last log lines:")
        print(''.join(open(log).readlines()[-12:])); return
    m = parse_results(f'/kaggle/working/runs/{name}'); m.update({'config':config,'fold':fold})
    save_row(m)
    print(f"  DONE {name}: mAP50={m['mAP50']:.3f} R={m['recall']:.3f} P={m['precision']:.3f} F1={m['f1']:.3f}")

def run_fold(fold):
    print(f"=== FOLD {fold} ===")
    run_one('baseline', fold)
    run_one('ga', fold)
    print(f"=== fold {fold} complete ===")

print("Runner ready (GA hyp = final values, mAP=0.674).")

## Restore of the folds completed in earlier sessions

The values below are transcribed from the run logs of the earlier sessions. The skip logic in
`run_one` reads them and resumes at the first fold not yet recorded.

In [ ]:
import pandas as pd, shutil, os
already_run = [
    ('baseline',   0, 0.496, 0.490, 0.637, 0.554),
    ('ga',         0, 0.729, 0.697, 0.827, 0.756),
    ('baseline',   1, 0.255, 0.391, 0.511, 0.443),
    ('ga',         1, 0.373, 0.367, 0.637, 0.466),
    ('baseline',   2, 0.446, 0.465, 0.507, 0.485),
    ('ga',         2, 0.458, 0.387, 0.616, 0.476),
    ('baseline',   3, 0.502, 0.392, 0.813, 0.529),
    ('ga',         3, 0.697, 0.594, 0.812, 0.686),
    ('baseline',   4, 0.581, 0.539, 0.751, 0.628),
    ('ga',         4, 0.656, 0.555, 0.663, 0.604),
    ('baseline',   5, 0.390, 0.555, 0.518, 0.536),
    ('ga',         5, 0.477, 0.536, 0.715, 0.613),
    ('baseline',   6, 0.441, 0.399, 0.616, 0.484),
    ('ga',         6, 0.644, 0.591, 0.732, 0.654),
    ('baseline',   7, 0.441, 0.413, 0.683, 0.514),
    ('ga',         7, 0.556, 0.438, 0.896, 0.588),
    ('baseline',   8, 0.412, 0.406, 0.610, 0.487),
    ('ga',         8, 0.615, 0.566, 0.719, 0.633),
    ('baseline',   9, 0.564, 0.537, 0.708, 0.611),
    ('ga',         9, 0.584, 0.461, 0.865, 0.602),
    ('baseline',  10, 0.283, 0.393, 0.266, 0.317),
    ('ga',        10, 0.302, 0.347, 0.772, 0.479),
    ('baseline',  11, 0.563, 0.558, 0.779, 0.650),
    ('ga',        11, 0.470, 0.521, 0.504, 0.512),
    ('baseline',  12, 0.356, 0.330, 0.404, 0.364),
    ('ga',        12, 0.656, 0.634, 0.736, 0.681),
    ('baseline',  13, 0.488, 0.450, 0.533, 0.488),
    ('ga',        13, 0.636, 0.593, 0.812, 0.686),
    ('baseline',  14, 0.360, 0.393, 0.354, 0.372),
    ('ga',        14, 0.451, 0.461, 0.459, 0.460),
    ('baseline',  15, 0.214, 0.324, 0.289, 0.306),
    ('ga',        15, 0.301, 0.367, 0.371, 0.369),
    ('baseline',  16, 0.487, 0.345, 0.804, 0.483),
    ('ga',        16, 0.663, 0.633, 0.738, 0.682),
    ('baseline',  17, 0.498, 0.496, 0.635, 0.557),
    ('ga',        17, 0.589, 0.595, 0.647, 0.620),
    ('baseline',  18, 0.398, 0.405, 0.513, 0.453),
    ('ga',        18, 0.431, 0.386, 0.554, 0.455),
    ('baseline',  19, 0.264, 0.292, 0.414, 0.342),
    ('ga',        19, 0.423, 0.410, 0.592, 0.485),
    ('baseline',  20, 0.210, 0.286, 0.342, 0.311),
    ('ga',        20, 0.313, 0.371, 0.378, 0.375),
    ('baseline',  21, 0.555, 0.584, 0.529, 0.556),
    ('ga',        21, 0.722, 0.593, 0.770, 0.670),
    ('baseline',  22, 0.520, 0.454, 0.741, 0.563),
    ('ga',        22, 0.529, 0.404, 0.903, 0.558),
    ('baseline',  23, 0.466, 0.502, 0.585, 0.540),
    ('ga',        23, 0.663, 0.636, 0.755, 0.691),
    ('baseline',  24, 0.526, 0.382, 0.677, 0.488),
    ('ga',        24, 0.656, 0.546, 0.693, 0.611),
    ('baseline',  25, 0.266, 0.261, 0.350, 0.299),
    ('ga',        25, 0.320, 0.327, 0.436, 0.374),
    ('baseline',  26, 0.314, 0.302, 0.557, 0.392),
    ('ga',        26, 0.359, 0.455, 0.378, 0.413),
    ('baseline',  27, 0.405, 0.329, 0.614, 0.429),
    ('ga',        27, 0.520, 0.403, 0.824, 0.541),
    ('baseline',  28, 0.603, 0.521, 0.716, 0.603),
    ('ga',        28, 0.699, 0.742, 0.612, 0.671),
    ('baseline',  29, 0.431, 0.417, 0.657, 0.510),
    ('ga',        29, 0.544, 0.417, 0.840, 0.557),
    ('baseline',  30, 0.357, 0.365, 0.623, 0.461),
    ('ga',        30, 0.426, 0.402, 0.646, 0.496),
    ('baseline',  31, 0.386, 0.339, 0.712, 0.459),
    ('ga',        31, 0.542, 0.526, 0.563, 0.544),
    ('baseline',  32, 0.475, 0.412, 0.654, 0.506),
    ('ga',        32, 0.486, 0.462, 0.606, 0.524),
    ('baseline',  33, 0.212, 0.307, 0.320, 0.313),
    ('ga',        33, 0.302, 0.293, 0.498, 0.369),
    ('baseline',  34, 0.490, 0.444, 0.598, 0.509),
    ('ga',        34, 0.468, 0.474, 0.730, 0.575),
    ('baseline',  35, 0.514, 0.459, 0.759, 0.572),
    ('ga',        35, 0.537, 0.420, 0.786, 0.547),
    ('baseline',  36, 0.521, 0.554, 0.661, 0.602),
    ('ga',        36, 0.721, 0.630, 0.723, 0.673),
]
rows = [{'config':c, 'fold':f, 'mAP50':m, 'recall':r,
         'precision':p, 'f1':f1, 'mAP50_95':0.0}
        for (c,f,m,r,p,f1) in already_run]
pd.DataFrame(rows).to_csv(results_path, index=False)
shutil.copy(results_path, DRIVE_BAK)
print(f'Restored {len(rows)} runs (folds 0-36). Επόμενο: fold 37.')

def run_fold_bk(i):
    run_fold(i)
    shutil.copy(results_path, DRIVE_BAK)
    d = f'/kaggle/working/data_fold{i}'
    if os.path.exists(d): shutil.rmtree(d)
    print(f'  backed up -> Drive ({len(pd.read_csv(results_path))} rows)')

## Run the remaining folds

In [ ]:
import time
assert len(folds) == 44, f"ΛΑΘΟΣ: {len(folds)} folds — διόρθωσε το Cell 3"

t0 = time.time()
for i in range(14, 44):
    run_fold_bk(i)
    n = len(pd.read_csv(results_path))
    print(f">>> Πρόοδος: {n}/88 runs | {(time.time()-t0)/3600:.1f}h\n")
print("===== ΟΛΑ ΤΑ FOLDS ΟΛΟΚΛΗΡΩΘΗΚΑΝ =====")

## Aggregate

In [ ]:
df = pd.read_csv(results_path)
print("Runs completed:", len(df), "/ 20\n")
for config in ['baseline', 'ga']:
    sub = df[df['config'] == config]
    if len(sub) == 0: continue
    print(f"{config.upper()}  (n={len(sub)} folds)")
    for metric in ['mAP50', 'recall', 'precision', 'f1', 'mAP50_95']:
        print(f"  {metric}: {sub[metric].mean():.3f} +/- {sub[metric].std():.3f}")
    print()

piv = df.pivot(index='fold', columns='config', values='mAP50')
if 'ga' in piv.columns and 'baseline' in piv.columns:
    piv['improvement'] = piv['ga'] - piv['baseline']
    print("Per-fold mAP50 improvement (GA - baseline):")
    print(piv.round(3))
    v = piv.dropna()
    if len(v):
        print(f"\nMean improvement: {v['improvement'].mean():.3f} +/- {v['improvement'].std():.3f} "
              f"(positive in {(v['improvement']>0).sum()}/{len(v)} folds)")